### Day 6
- Open project_dataset.csv in Excel; apply filters and conditional formatting
- Apply cleaning plan in Excel: fix types, trim whitespace, harmonise Production_Companies, Primary_Production_Company, Cast_Gender_Ratio
- Validate and reconcile row counts against original file

In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/project_dataset.csv")

df_original = df.copy()

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 4803
Columns: 51


### Trim Whitespace in Text Columns

In [17]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

whitespace_report = []

for col in text_columns:

    mask = (
        df[col].notna()
        & (df[col].astype(str) != df[col].astype(str).str.strip())
    )

    issue_count = int(mask.sum())

    if issue_count > 0:
        whitespace_report.append({
            "Column": col,
            "Whitespace_Issues_Fixed": issue_count
        })

    # Trim whitespace
    df[col] = df[col].apply(
        lambda x: x.strip()
        if isinstance(x, str)
        else x
    )

whitespace_report = pd.DataFrame(whitespace_report)

print("WHITESPACE CLEANING REPORT")

if whitespace_report.empty:
    print("No leading/trailing whitespace issues found.")
else:
    print(whitespace_report)

WHITESPACE CLEANING REPORT
No leading/trailing whitespace issues found.


### Fix Release_Date Data Type

In [18]:
if "Release_Date" in df.columns:

    df["Release_Date"] = pd.to_datetime(
        df["Release_Date"],
        errors="coerce",
        dayfirst=True
    )

    print("Release_Date Type:", df["Release_Date"].dtype)

    print(
        "Missing/Invalid Release_Date:",
        df["Release_Date"].isna().sum()
    )

Release_Date Type: datetime64[ns]
Missing/Invalid Release_Date: 8


### Fix and Validate Release_Year

In [19]:
if "Release_Year" in df.columns:

    df["Release_Year"] = pd.to_numeric(
        df["Release_Year"],
        errors="coerce"
    ).astype("Int64")

    print("Release_Year Type:", df["Release_Year"].dtype)

    print(
        "Missing/Invalid Release_Year:",
        df["Release_Year"].isna().sum()
    )

Release_Year Type: Int64
Missing/Invalid Release_Year: 8


### Harmonise Production_Companies, Primary_Production_Companies

In [20]:
if "Production_Companies" in df.columns:

    df["Production_Companies"] = (
        df["Production_Companies"]
        .apply(
            lambda x: " ".join(x.split())
            if isinstance(x, str)
            else x
        )
    )

    print(
        "Missing Production_Companies:",
        df["Production_Companies"].isna().sum()
    )

    print("\nTop 20 Values:")

    print(
        df["Production_Companies"]
        .value_counts(dropna=False)
        .head(20)
    )

Missing Production_Companies: 123

Top 20 Values:
Production_Companies
NaN                                                    123
Paramount Pictures                                      28
20th Century Fox                                        21
Metro-Goldwyn-Mayer                                     16
Columbia Pictures                                       16
Warner Bros. Pictures                                   15
Pixar                                                   14
DreamWorks Animation                                    13
Universal Pictures                                      12
Morgan Creek Entertainment                              11
New Line Cinema                                         10
Walt Disney Pictures                                    10
Touchstone Pictures                                      9
EON Productions, United Artists                          9
Marvel Studios                                           8
EON Productions                             

In [21]:
if "Primary_Production_Company" in df.columns:

    df["Primary_Production_Company"] = (
        df["Primary_Production_Company"]
        .apply(
            lambda x: " ".join(x.split())
            if isinstance(x, str)
            else x
        )
    )

    print(
        "Missing Primary Production Company:",
        df["Primary_Production_Company"].isna().sum()
    )

    print("\nTop 20 Values:")

    print(
        df["Primary_Production_Company"]
        .value_counts(dropna=False)
        .head(20)
    )

Missing Primary Production Company: 123

Top 20 Values:
Primary_Production_Company
Paramount Pictures           183
Universal Pictures           158
Columbia Pictures            131
NaN                          123
New Line Cinema              112
Warner Bros. Pictures         94
Walt Disney Pictures          74
20th Century Fox              66
Touchstone Pictures           65
Metro-Goldwyn-Mayer           51
Miramax                       47
DreamWorks Pictures           42
Lionsgate                     35
Fox Searchlight Pictures      34
United Artists                32
Fox 2000 Pictures             32
Castle Rock Entertainment     30
Dimension Films               30
TriStar Pictures              29
Screen Gems                   29
Name: count, dtype: int64


### Validate Cast_Gender_Ratio

In [22]:
if "Cast_Gender_Ratio" in df.columns:

    df["Cast_Gender_Ratio"] = pd.to_numeric(
        df["Cast_Gender_Ratio"],
        errors="coerce"
    )

    missing_ratio = (
        df["Cast_Gender_Ratio"]
        .isna()
        .sum()
    )

    zero_ratio = (
        df["Cast_Gender_Ratio"] == 0
    ).sum()

    negative_ratio = (
        df["Cast_Gender_Ratio"] < 0
    ).sum()

    print("CAST GENDER RATIO VALIDATION")

    print("Missing Values:", missing_ratio)
    print("Zero Values:", zero_ratio)
    print("Negative Values:", negative_ratio)

CAST GENDER RATIO VALIDATION
Missing Values: 307
Zero Values: 26
Negative Values: 0


### Save Cleaned Dataset

In [23]:
df.to_csv(
    "../data/cleaned/project_dataset_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")
print("File: data/cleaned/project_dataset_cleaned.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Cleaned dataset saved successfully.
File: data/cleaned/project_dataset_cleaned.csv
Rows: 4803
Columns: 51


## Validate and Reconcile Against Original

#### Row and Column Count Reconciliation

In [24]:
original_rows = df_original.shape[0]
cleaned_rows = df.shape[0]

original_columns = df_original.shape[1]
cleaned_columns = df.shape[1]

row_difference = cleaned_rows - original_rows
column_difference = cleaned_columns - original_columns

print("ROW AND COLUMN RECONCILIATION")

print("\nOriginal Rows:", original_rows)
print("Cleaned Rows :", cleaned_rows)
print("Row Difference:", row_difference)

print("\nOriginal Columns:", original_columns)
print("Cleaned Columns :", cleaned_columns)
print("Column Difference:", column_difference)

ROW AND COLUMN RECONCILIATION

Original Rows: 4803
Cleaned Rows : 4803
Row Difference: 0

Original Columns: 51
Cleaned Columns : 51
Column Difference: 0


#### Validate Column Structure

In [25]:
same_columns = (
    df_original.columns.tolist()
    == df.columns.tolist()
)

print("Same Columns and Order:", same_columns)

if not same_columns:

    print(
        "\nOnly in Original:",
        set(df_original.columns) - set(df.columns)
    )

    print(
        "Only in Cleaned:",
        set(df.columns) - set(df_original.columns)
    )

Same Columns and Order: True


#### Validate movie_id

In [26]:
if "movie_id" in df.columns:

    original_duplicate_ids = (
        df_original["movie_id"]
        .duplicated()
        .sum()
    )

    cleaned_duplicate_ids = (
        df["movie_id"]
        .duplicated()
        .sum()
    )

    original_ids = set(
        df_original["movie_id"].dropna()
    )

    cleaned_ids = set(
        df["movie_id"].dropna()
    )

    same_ids = original_ids == cleaned_ids

    print("MOVIE ID VALIDATION")

    print(
        "Original Duplicate movie_id:",
        original_duplicate_ids
    )

    print(
        "Cleaned Duplicate movie_id:",
        cleaned_duplicate_ids
    )

    print(
        "Same movie_id Set:",
        same_ids
    )

MOVIE ID VALIDATION
Original Duplicate movie_id: 0
Cleaned Duplicate movie_id: 0
Same movie_id Set: True


#### Compare Missing Values Before vs After Cleaning

In [27]:
missing_comparison = pd.DataFrame({

    "Original_Missing":
        df_original.isnull().sum(),

    "Cleaned_Missing":
        df.isnull().sum()
})

missing_comparison["Difference"] = (
    missing_comparison["Cleaned_Missing"]
    - missing_comparison["Original_Missing"]
)

missing_affected = missing_comparison[
    (missing_comparison["Original_Missing"] > 0)
    |
    (missing_comparison["Cleaned_Missing"] > 0)
]

print("MISSING VALUE COMPARISON")

print(missing_affected)

MISSING VALUE COMPARISON
                            Original_Missing  Cleaned_Missing  Difference
Budget                                   564              564           0
Revenue                                  736              736           0
Popularity                                 8                8           0
Genres                                    10               10           0
Director                                  30               30           0
Production_Companies                     123              123           0
Primary_Production_Company               123              123           0
Release_Date                               8                8           0
Lead_Actor                                43               43           0
Second_Lead_Actor                         53               53           0
Third_Lead_Actor                          63               63           0
Cast_Gender_Ratio                        307              307           0
Profit       

#### Check Duplicate Rows

In [28]:
original_duplicates = (
    df_original.duplicated().sum()
)

cleaned_duplicates = (
    df.duplicated().sum()
)

print("DUPLICATE ROW VALIDATION")

print(
    "Original Duplicate Rows:",
    original_duplicates
)

print(
    "Cleaned Duplicate Rows:",
    cleaned_duplicates
)

DUPLICATE ROW VALIDATION
Original Duplicate Rows: 0
Cleaned Duplicate Rows: 0


#### Final Reconciliation Status

In [29]:
row_match = (
    df_original.shape[0] == df.shape[0]
)

column_match = (
    df_original.shape[1] == df.shape[1]
)

column_structure_match = (
    df_original.columns.tolist()
    == df.columns.tolist()
)

if "movie_id" in df.columns:

    id_match = (
        set(df_original["movie_id"].dropna())
        ==
        set(df["movie_id"].dropna())
    )

else:
    id_match = True


if (
    row_match
    and column_match
    and column_structure_match
    and id_match
):

    status = "RECONCILED"

else:

    status = "REVIEW REQUIRED"


print("=" * 60)
print("FINAL DATA RECONCILIATION")
print("=" * 60)

print("Original Rows      :", df_original.shape[0])
print("Cleaned Rows       :", df.shape[0])
print("Row Difference     :", df.shape[0] - df_original.shape[0])

print()

print("Original Columns   :", df_original.shape[1])
print("Cleaned Columns    :", df.shape[1])
print(
    "Column Difference  :",
    df.shape[1] - df_original.shape[1]
)

print()

print("Column Structure OK:", column_structure_match)
print("Movie IDs Preserved:", id_match)

print()

print("FINAL STATUS:", status)

FINAL DATA RECONCILIATION
Original Rows      : 4803
Cleaned Rows       : 4803
Row Difference     : 0

Original Columns   : 51
Cleaned Columns    : 51
Column Difference  : 0

Column Structure OK: True
Movie IDs Preserved: True

FINAL STATUS: RECONCILED
